STEP - 1 - Simple Document

In [3]:
document = '''
A simple script to test the functionality of the `pyinstaller` package. 
It is not intended to be a comprehensive test, but rather a quick check to ensure that the package is working as expected.
'''
print(document)


A simple script to test the functionality of the `pyinstaller` package. 
It is not intended to be a comprehensive test, but rather a quick check to ensure that the package is working as expected.



STEP - 2 - Chunking

In [4]:
text = '''
A simple script to test the functionality of the `pyinstaller` package. 
It is not intended to be a comprehensive test, but rather a quick check to ensure that the package is working as expected.
'''
chunk = text.split(".")
print(chunk)

['\nA simple script to test the functionality of the `pyinstaller` package', ' \nIt is not intended to be a comprehensive test, but rather a quick check to ensure that the package is working as expected', '\n']


STEP - 3 - Remove Empty Chunks

In [5]:
chunk = [chunk.strip() for chunk in chunk if chunk.strip()]
print(chunk)

['A simple script to test the functionality of the `pyinstaller` package', 'It is not intended to be a comprehensive test, but rather a quick check to ensure that the package is working as expected']


STEP - 4 - Install Sentence Transformers

In [6]:
!pip3 install sentence-transformers

   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 6.8 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP - 5 -Understanding Embeddings

In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence = "This is an example sentence."
embedding = model.encode(sentence)
print(embedding.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7552.28it/s]


(384,)


STEP - 6 - Embedding Multiple Chunks

In [9]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk = [
    "This is the first sentence.",
    "This is the second sentence.",
    "This is the third sentence."
]
embeddings = model.encode(chunk)
print(embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7580.91it/s]


(3, 384)


STEP - 7 - User Query

In [10]:
query = "What is the second sentence?"
print(query)

What is the second sentence?


STEP - 8 - Query Embedding

In [11]:
query_embedding = model.encode(query)
print(query_embedding.shape)
print(query_embedding)

(384,)
[ 5.05856648e-02  1.25730783e-01  3.05072702e-02  1.03218600e-01
  3.37601155e-02  4.44885865e-02  1.33675039e-01  1.47426696e-02
 -6.39271783e-03 -6.24356680e-02  1.45248517e-01 -5.32143638e-02
  7.02512637e-03 -1.17958583e-01  9.82460566e-03  4.40864004e-02
 -4.11314480e-02 -4.25524749e-02 -4.99492660e-02  6.65327627e-03
  4.80694510e-02 -5.74733578e-02  2.49353144e-02  2.24594530e-02
  2.30804854e-03  1.00694761e-01 -4.36645672e-02  7.36105666e-02
  3.65836397e-02  6.83619129e-03 -2.77482383e-02  4.76073334e-03
 -2.27661133e-02  6.26483932e-02 -2.31392812e-02 -2.60861870e-02
  2.11989209e-02  1.58216991e-02  2.47065863e-03 -4.56750579e-02
  1.46247009e-02 -7.35099018e-02  5.85684832e-03 -1.37019577e-02
 -1.73644684e-02 -2.16980074e-02 -1.83882583e-02  4.95337741e-03
 -3.19986828e-02 -4.83058430e-02 -1.06886931e-01 -3.33514884e-02
 -8.61401036e-02  1.66901536e-02  2.03747656e-02  8.63263607e-02
 -2.27237307e-02  6.94426224e-02  1.27642648e-02 -3.16949263e-02
  2.22626142e-02 -

STEP - 10 - Similarity martics

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
similarity= cosine_similarity([query_embedding], embeddings)
print(similarity)

[[0.67344844 0.74437106 0.71605146]]


STEP - 11 - Retrieve Best Matching chunk

In [13]:
best_match_index = similarity.argmax()
print(chunk[best_match_index])

This is the second sentence.


STEP - 12 - Prompt Augmentation

In [15]:
retrieved_chunk = chunk[best_match_index]
prompt = f'''
Context: 
{retrieved_chunk}
Question: 
{query}
'''
print(prompt)


Context: 
This is the second sentence.
Question: 
What is the second sentence?



STEP - 13 - Response

In [16]:
answer = "The second sentence is: This is the second sentence."
print(answer)

The second sentence is: This is the second sentence.


STEP - 14 - Complete Mini RAG Pipeline

In [18]:

import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
documents = [
    """Retrieval-Augmented Generation (RAG) combines retrieval and text generation.
    First, relevant chunks are fetched from a document store.
    Then, the LLM uses those chunks as context to answer questions.""",

    """Chunking splits large documents into smaller overlapping text blocks.
    Overlap helps preserve meaning across chunk boundaries and improves recall.""",

    """Embeddings are dense vector representations of text.
    Similar meaning -> closer vectors in embedding space.
    Cosine similarity is commonly used for retrieval."""
]
def chunk_text(text, chunk_size=220, overlap=40):
    text = re.sub(r"\s+", " ", text).strip()
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks
chunk_store = []
for doc_id, doc in enumerate(documents):
    chunks = chunk_text(doc, chunk_size=220, overlap=40)
    for i, ch in enumerate(chunks):
        chunk_store.append({
            "doc_id": doc_id,
            "chunk_id": i,
            "text": ch
        })

print(f"Total chunks: {len(chunk_store)}")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in chunk_store]
chunk_embeddings = embedder.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)

def retrieve(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        item = chunk_store[idx].copy()
        item["score"] = float(sims[idx])
        results.append(item)
    return results
def generate_answer(query, retrieved_chunks):
    context = "\n".join([f"- {r['text']}" for r in retrieved_chunks])
    answer = (
        f"Question: {query}\n\n"
        f"Answer (grounded in retrieved context):\n"
        f"{context}\n\n"
        f"Summary: Based on the retrieved passages, RAG retrieves relevant chunks using embeddings "
        f"and similarity search, then uses that context to produce an answer."
    )
    return answer
def rag_pipeline(query, top_k=3, show_context=True):
    retrieved = retrieve(query, top_k=top_k)
    if show_context:
        print("Top retrieved chunks:")
        for i, r in enumerate(retrieved, 1):
            print(f"{i}. score={r['score']:.4f} | doc={r['doc_id']} chunk={r['chunk_id']}")
            print(f"   {r['text']}\n")
    return generate_answer(query, retrieved)

query = "Why do we use chunk overlap in RAG?"
final_answer = rag_pipeline(query, top_k=3, show_context=True)
print(final_answer)

Total chunks: 4


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7758.86it/s]


Top retrieved chunks:
1. score=0.4957 | doc=1 chunk=0
   Chunking splits large documents into smaller overlapping text blocks. Overlap helps preserve meaning across chunk boundaries and improves recall.

2. score=0.4664 | doc=0 chunk=0
   Retrieval-Augmented Generation (RAG) combines retrieval and text generation. First, relevant chunks are fetched from a document store. Then, the LLM uses those chunks as context to answer questions.

3. score=0.1570 | doc=2 chunk=0
   Embeddings are dense vector representations of text. Similar meaning -> closer vectors in embedding space. Cosine similarity is commonly used for retrieval.

Question: Why do we use chunk overlap in RAG?

Answer (grounded in retrieved context):
- Chunking splits large documents into smaller overlapping text blocks. Overlap helps preserve meaning across chunk boundaries and improves recall.
- Retrieval-Augmented Generation (RAG) combines retrieval and text generation. First, relevant chunks are fetched from a document sto

Side Quest -> LangChain

In [17]:
!pip3 install langchain

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 41.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 23.2 MB/s  0:00:00

   ---------- -----------------------------  5/19 [orjson]
   ------------------ ---------------------  9/19 [requests-toolbelt]
   --------------------- ------------------ 10/19 [pydantic]
   --------------------- ------------------ 10/19 [pydantic]
   --------------------- ------------------ 10/19 [pydantic]
   ------------------------- -------------- 12/19 [langsmith]
   ------------------------- -------------- 12/19 [langsmith]
   ------------------------- -------------- 12/19 [langsmith]
   --------------------------- ------------ 13/19 [langgraph-sdk]
   ----------------------------- ---------- 14/19 [langchain-core]
   ----------------------------- ---------- 14/19 [langchain-core]
   ------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP - 15 - Langchain chunking

In [19]:
!pip3 install langchain-text-splitters


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from langchain_text_splitters import CharacterTextSplitter
text = "This is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split.this is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split."
text_splitter = CharacterTextSplitter(chunk_size=30, chunk_overlap=5)
chunks = text_splitter.split_text(text)
print(chunks)

['This is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split.this is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split.']


STEP - 16 - FAISS vector DB

In [24]:
!pip3 install langchain-community langchain-huggingface

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 11.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 23.7 MB/s  0:00:00

   ---------- -----------------------------  3/11 [greenlet]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   ------------------ ---------------------  5/11 [SQLAlchemy]
   --


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
!pip3 install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.9 MB 14.0 MB/s eta 0:00:02
   ---------- ----------------------------- 5.0/18.9 MB 18.2 MB/s eta 0:00:01
   -------------------- ------------------- 9.7/18.9 MB 20.4 MB/s eta 0:00:01
   ------------------------------ --------- 14.4/18.9 MB 22.5 MB/s eta 0:00:01
   ------------------------------------ --- 17.3/18.9 MB 22.3 MB/s eta 0:00:01
   ------------------------------------ --- 17.3/18.9 MB 22.3 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 15.8 MB/s  0:00:01



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings()
db = FAISS.from_texts(chunks, embedding_model)

c:\Users\sanja\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanja\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5128.49it/s]


STEP - 17 - Retrieval using FAISS

In [ ]:
result = db.similarity_search("long document")
print(result[0].page_content)

This is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split.this is a long document that needs to be split into smaller chunks for better processing.And this is another sentence to ensure we have enough text to split.
